# Feature Selection

Feature selection picks the subset of features that carry the most signal for a target variable.
Fewer, better features mean faster training, lower memory footprint, and simpler models.

## Techniques Covered

- **Feature importance** (linear weights, XGBoost gain) — rank all features by predictive power
- **Minimal-feature model** — verify that top-2 features alone capture most of the signal
- **Random column baseline** — if a random noise column beats a real feature, drop the real feature
- **Recursive Feature Elimination (RFE)** — iteratively prune lowest-ranked features

## Dataset

EPA Fuel Economy (`vehicles.csv`) — same dataset used across `ml-features/`.
Target: `city08` (city MPG).


In [ ]:
# Install dependencies (run once)
# scikit-learn ships as 'scikit-learn'; 'sklearn' is not a valid pip package
!pip install -q scikit-learn xgboost category_encoders

In [ ]:
# Resolve utils.py from the notebook directory regardless of kernel launch path
import sys, pathlib
_vsc_nb = globals().get('__vsc_ipynb_file__')
_nb_dir = pathlib.Path(_vsc_nb).parent if _vsc_nb else pathlib.Path.cwd()
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

import pandas as pd
import numpy as np
from utils import (
    load_fuel_economy_data,
    debug_transformer,
    combine_str_cols_transformer,
    TextPipeline,
    SpacyVectorizer,
    CAT_COLS, LOW_CARDINALITY_COLS, HIGH_CARDINALITY_COLS,
    make_preprocessor,
)

In [ ]:
# analyzing the data
df = load_fuel_economy_data()

df.head()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# Drop correlated mileage targets and the raw timezone string to avoid leakage
X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# Baseline: full preprocessing pipeline (TF-IDF + OHE + target encoding) -> linear regression
# make_preprocessor() is defined in utils.py
pipeline = Pipeline(steps=[
    ('preprocessor', make_preprocessor()),
    ('lr', LinearRegression()),
])
pipeline.fit(X_train, y_train)

In [ ]:
# Inspect the fitted LinearRegression step extracted from the pipeline
lr = pipeline.named_steps['lr']
lr

In [ ]:
# LinearRegression feature importance via coefficient magnitudes
# feature_names_in_ requires pandas output mode (set_config below)
from sklearn import set_config
set_config(transform_output='pandas')  # make transformers return DataFrames

# Re-fit so feature names propagate through the pipeline
pipeline.fit(X_train, y_train)
lr = pipeline.named_steps['lr']

# Show top 25 features by absolute coefficient weight
(pd.Series(lr.coef_, index=lr.feature_names_in_)
 .sort_values(key=abs)
 .tail(25)
 .plot.barh(title='Top 25 features by absolute coefficient'))

## XGBoost Feature Importance

XGBoost works natively with categorical features via `enable_categorical=True`
and exposes `feature_importances_` based on **gain** — the average loss reduction
from all splits on that feature. Unlike linear coefficients, gain captures
non-linear and interaction effects.


In [ ]:
# xgboost is a much more powerful option to linear regression
# It can work with any shape of the feature's relationship with target values
# Requires much less feature engineering since it automatically ignores the weaker relationships (we cap the number of levels)
# It however doesn't always give a concrete picture of feature importance which may be important
# when auditing and analyzing the model over the performance of the model

# rfe (recursive feature elimination) works best with xgboost
import xgboost as xgb


# create X and y
X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X.assign(**X.select_dtypes(object).astype('category')), y, random_state=42)

'''
code breakdown
X.assign(**X.select_dtypes(object).astype('category'))
'category' and 'object' are types in py
'category' type is a bit like enum, they are powered by low-level ints even though they represent string values
this makes the encoding and computation faster
In Python, if you have a function that expects named arguments, you can pass a dictionary to it using **.
It "unpacks" the keys into argument names and the values into the argument values.
Eg.

def my_function(name, role):
    print(f"{name} is a {role}")

data = {"name": "ds", "role": "Engineer"}

# This:
my_function(**data)

# Is exactly the same as typing this:
my_function(name="ds", role="Engineer")


The Pandas .assign() method is designed to take any number of keyword arguments. For example:
df.assign(City='Seattle', State='WA')

When you run X.select_dtypes(object).astype('category'), it returns a mini-DataFrame. In Python's eyes, a DataFrame
can behave like a collection of Series where the column names are keys and the columns themselves are values.
'column_name':{'a':'1', 'b':'2', 'c':'3'}
'''


xg = xgb.XGBRegressor(enable_categorical=True , random_state=42)
xg.fit(X_train, y_train)
xg.score(X_test, y_test)

In [ ]:
# Plot feature importances; .iloc[:-1] excludes the random column if present
pd.Series(xg.feature_importances_, index=X_train.columns).sort_values().iloc[:-1].plot.barh()

In [ ]:
from xgboost import XGBRegressor
# now select only the super important features from the bar above i.e., barrels08, fuelCost08

X = df[['barrels08', 'fuelCost08']]
y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X.assign(**X.select_dtypes(object).astype('category')), y, random_state=42)


xg_minimal = XGBRegressor()
xg_minimal.fit(X_train, y_train)
xg_minimal.score(X_test, y_test)

In [ ]:
# Random Column addition
# If a randomly generated column  performs better than a feature, the feature is bound to be unimportant
# NOTE: A feature unimportant in Xgboost may still be relevant for linear regression, i.e., feature importance
# is tied to the type of model we are planning to use

# create X and y
X = (df
    .drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
    .assign(random=lambda df: np.random.random(size=len(df))))

y = df.city08

X_train, X_test, y_train, y_test = train_test_split(X.assign(**X.select_dtypes(object).astype('category')), y, random_state=42)


xg = xgb.XGBRegressor(enable_categorical=True , random_state=42)
xg.fit(X_train, y_train)

In [ ]:
# After adding a random noise column:
# any real feature ranked BELOW the random column carries no useful signal
pd.Series(xg.feature_importances_, index=X_train.columns).sort_values().plot.barh(
    title='Feature importance vs random baseline')

In [ ]:
from sklearn.feature_selection import RFE

# use rfe with xgboost
xg_model = xgb.XGBRegressor(enable_categorical=True, random_state=42)
rfe = RFE(xg_model, n_features_to_select=3)

X = df.drop(columns=['city08', 'highway08', 'comb08', 'createdOn'])
y = df.city08

# split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    #X.assign(**X.select_dtypes('string[pyarrow]').astype('category')),
    X.select_dtypes('number'),
    y, random_state=42)

rfe.fit(X_train, y_train)
# Explore the API to determine which features were selected

pd.DataFrame({'features':X_train.columns,
              'support': rfe.support_,
              'ranking':rfe.ranking_}).sort_values('ranking')

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

X=df.select_dtypes('number').drop(columns=['city08']).dropna(axis=1)
y=df.city08

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

lr = LinearRegression()


lr.fit(X_train, y_train)
print(f"R2 score is {lr.score(X_test, y_test)}")
print(f"mse is {mean_absolute_error(y_test, lr.predict(X_test))}")


rfe = RFE(lr, n_features_to_select=3)
rfe.fit(X_train, y_train)

pd.DataFrame({'features':X_train.columns,
              'support': rfe.support_,
              'ranking':rfe.ranking_}).sort_values('ranking')